# Kapitel 7: Robuste CVaR-Modelle in der Finanzkrise 2007–2009

Dieses Notebook erzeugt alle Zahlen und Tabellen aus Kapitel 7 der Arbeit. Grundlage sind die
wertgewichteten Tagesrenditen der 30 Industry Portfolios vom 02.07.2007 bis zum 31.12.2009
(CRSP-Version 202608).

**Pakete:** `numpy`, `pandas`, `scipy`, `cvxpy` (Solver `ECOS`, falls nicht installiert `CLARABEL`).

**Ausführung:** Datenpfad über `CSV_PATH` setzen und alle Zellen von oben nach unten ausführen
(*Run All*).

**Zuordnung zur Arbeit**

| Notebook | Arbeit |
|---|---|
| 1–3 | Abschnitt 7.1, Tabellen zu den Marktphasen und zur Normalitätsprüfung |
| 4–5 | Abschnitt 7.2.1 |
| 6 | Abschnitt 7.2.2 |
| 7 | Abschnitt 7.3.1, Tabellen zum 1/n-Portfolio, zu SOCP und LP und zu den Portfoliogewichten |
| 8 | Abschnitt 7.3.2, Tabelle zum Regret je Phase, CVaR der einzelnen Branchen |
| 9 | Robustheit gegenüber den Phasengrenzen und gleich lange Blöcke (Abschnitte 7.3.1, 7.3.2 und 8.3) |


## 1. Daten laden (Abschnitt 7.1.1)

Wertgewichtete Tagesrenditen in Prozent im Zeitraum vom 02.07.2007 bis zum 31.12.2009. Die erste
Zeile der Datei nennt die CRSP-Version.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm, jarque_bera, kurtosis
import cvxpy as cp

CSV_PATH = "30_Industry_Portfolios_Daily.csv"
BLOCK_NAME = "Average Value Weighted Returns -- Daily"


def load_30_industry_portfolios_daily(csv_path, start=20070701, end=20091231):
    """Liest den wertgewichteten Tagesblock der Kenneth-French-CSV und filtert auf [start, end]"""
    with open(csv_path, "r") as f:
        lines = f.readlines()
    header_idx = next(i for i, l in enumerate(lines) if BLOCK_NAME in l)
    colnames_idx = header_idx + 1
    while lines[colnames_idx].strip() == "":
        colnames_idx += 1
    colnames = [c.strip() for c in lines[colnames_idx].strip().split(",")]
    colnames[0] = "date"
    data_rows = []
    i = colnames_idx + 1
    while i < len(lines) and lines[i].strip() != "":
        data_rows.append([v.strip() for v in lines[i].strip().split(",")])
        i += 1
    df = pd.DataFrame(data_rows, columns=colnames)
    df["date"] = df["date"].astype(int)
    df = df.set_index("date").astype(float)
    df = df.loc[(df.index >= start) & (df.index <= end)].replace([-99.99, -999], np.nan)
    assert not df.isna().any().any(), "Fehlende Werte im Untersuchungszeitraum"
    return df


df = load_30_industry_portfolios_daily(CSV_PATH)
with open(CSV_PATH) as f:
    print(f.readline().strip())
print("Shape:", df.shape, "| Zeitraum:", df.index.min(), "-", df.index.max())

df.head()

This file was created by CMPT_IND_RETS using the 202608 CRSP database.
Shape: (632, 30) | Zeitraum: 20070702 - 20091231


,Food,Beer,Smoke,Games,Books,Hshld,Clths,Hlth,Chems,Txtls,...,Telcm,Servs,BusEq,Paper,Trans,Whlsl,Rtail,Meals,Fin,Other
date,,,,,,,,,,,,,,,,,,,,,
20070702,1.04,0.54,0.96,0.40,0.62,0.81,0.40,0.88,1.66,0.88,...,1.42,0.60,1.06,1.21,0.95,0.94,0.66,0.79,1.19,0.42
20070703,-0.64,0.19,0.21,-0.38,-0.29,0.28,0.44,0.19,0.17,-0.03,...,0.21,0.52,0.78,0.11,0.75,0.11,-0.07,1.21,0.58,0.73
20070705,-0.39,0.14,0.35,2.12,0.62,-0.07,1.10,0.09,0.76,0.71,...,-0.22,0.51,0.57,0.42,-0.03,0.07,0.12,2.93,-0.49,0.00
20070706,0.17,0.01,0.57,0.53,0.10,0.07,1.16,-0.11,0.31,0.35,...,0.33,0.32,0.44,-0.01,0.51,0.03,1.24,0.33,0.37,0.11
20070709,0.64,-0.16,-0.17,0.22,-0.33,0.21,-0.32,-0.05,0.51,-1.03,...,-0.20,-0.10,0.19,-0.03,0.93,0.07,-0.32,-0.51,-0.36,0.29


## 2. Marktphasen (Abschnitt 7.1.1)

Die Grenzen bilden die Insolvenz von Lehman Brothers am 15.09.2008 und der Tiefpunkt des S&P 500
am 09.03.2009.

In [2]:
PHASES = [
    ("Vorkrise", 20070702, 20080912),
    ("Crash",    20080915, 20090309),
    ("Erholung", 20090310, 20091231),
]
dates = df.index.to_numpy()
masks = {name: (dates >= a) & (dates <= b) for name, a, b in PHASES}
T_phase = {name: int(m.sum()) for name, m in masks.items()}
for name, a, b in PHASES:
    d = dates[masks[name]]
    print(f"{name:<9}: {d.min()} - {d.max()}  ({T_phase[name]} Handelstage)")

Vorkrise : 20070702 - 20080912  (304 Handelstage)
Crash    : 20080915 - 20090309  (121 Handelstage)
Erholung : 20090310 - 20091231  (207 Handelstage)


## 3. Normalitätsprüfung (Abschnitt 7.1.2)

Für jede Branche wird der Jarque-Bera-Test durchgeführt, einmal für die gesamte Krise und einmal
je Phase. Zusätzlich werden die mittlere Exzess-Kurtosis (bei Normalverteilung null) und die
mittlere Tagesvolatilität berechnet

In [3]:
def normality_summary(block):
    pvals = np.array([jarque_bera(block[c]).pvalue for c in block.columns])
    return {
        "Handelstage": len(block),
        "JB verworfen (5 %)": f"{np.mean(pvals < 0.05):.1%}",
        "Ø Exzess-Kurtosis": round(float(np.mean(kurtosis(block, axis=0))), 2),
        "Ø Tagesvolatilität": round(float(block.std().mean()), 2),
    }


normality_df = pd.DataFrame(
    {"Krise gesamt": normality_summary(df),
     **{name: normality_summary(df[m]) for name, m in masks.items()}}
).T
normality_df

,Handelstage,JB verworfen (5 %),Ø Exzess-Kurtosis,Ø Tagesvolatilität
Krise gesamt,632,100.0%,4.3,2.44
Vorkrise,304,50.0%,0.52,1.66
Crash,121,33.3%,0.78,4.1
Erholung,207,83.3%,1.96,2.01


## 4. Modellaufbau:  Marktphasen als Experten (Abschnitt 7.2.1)

Jeder Handelstag $k = 1, \dots, S$ ist ein Szenario mit dem Renditevektor $r_{k}$. Phase $i$
liefert eine Expertenverteilung, die nur die eigenen Handelstage gleich gewichtet,

$$
\pi^{i}_{k} = \begin{cases} 1/T_{i}, & \text{wenn Tag } k \text{ zu Phase } i \text{ gehört}, \\ 0, & \text{sonst}, \end{cases}
\qquad
\pi^{0}_{k} = \frac{1}{S}.
$$

Wegen $T_{1} + T_{2} + T_{3} = S$ ist die Nominalverteilung die Mischung
$\pi^{0} = \sum_{i} \frac{T_{i}}{S}\, \pi^{i}$ der Expertenverteilungen 

In [4]:
R = df.to_numpy()
S, n = R.shape
eps = 0.95

pi_0 = np.full(S, 1.0 / S)
pi_experts = {}
for name, m in masks.items():
    pi_i = np.zeros(S)
    pi_i[m] = 1.0 / m.sum()
    pi_experts[name] = pi_i
pi_list = list(pi_experts.values())


## 5. Modelle (Abschnitt 7.2.1)

### 5.1 Empirischer CVaR und Normalformel

Für eine Verteilung $\pi$ über die Szenarien gilt nach Rockafellar und Uryasev

$$
\mathrm{CVaR}_{\epsilon}(x, \pi) = \min_{\gamma \in \mathbb{R}} F_{\epsilon}(x, \gamma, \pi),
\qquad
F_{\epsilon}(x, \gamma, \pi) = \gamma + \frac{1}{1-\epsilon} \sum_{k=1}^{S} \pi_{k} \left[-r_{k}^{\top} x - \gamma\right]^{+}.
$$

Ein Minimierer ist der VaR, also kleinste $\gamma$ mit $\pi(\text{Verlust} \leq \gamma) \geq \epsilon$.
Die Funktion `cvar_discrete` sortiert die Verluste, bestimmt diesen VaR und wertet $F_{\epsilon}$
dort aus. Bei $\epsilon = 0{,}95$ ist das der mittlere Verlust der schlechtesten 5 % der
gewichteten Handelstage. Zur Kontrolle wird das Ergebnis einmal mit dem linearen Programm
$\min_{\gamma} F_{\epsilon}$ verglichen.

Die Funktion `cvar_normal` berechnet die geschlossene Formel aus Abschnitt 5.3 der Arbeit mit den
Stichprobenmomenten $\hat{r}$ und $\hat{\Sigma}$ desselben Zeitraums,

$$
\mathrm{CVaR}^{\mathrm{N}}_{\epsilon}(x) = -x^{\top} \hat{r} + k_{\epsilon} \sqrt{x^{\top} \hat{\Sigma} x},
\qquad
k_{\epsilon} = \frac{\varphi(\Phi^{-1}(\epsilon))}{1-\epsilon}.
$$

In [5]:
SOLVER = cp.ECOS if "ECOS" in cp.installed_solvers() else cp.CLARABEL
print("Solver:", SOLVER)


def cvar_discrete(losses, probs, eps=0.95):
    """CVaR_eps einer diskreten Verlustverteilung: F_eps(gamma*) mit gamma* = VaR_eps"""
    order = np.argsort(losses)
    cum = np.cumsum(probs[order])
    gamma = losses[order][np.searchsorted(cum, eps - 1e-12)]
    return gamma + probs @ np.maximum(losses - gamma, 0) / (1 - eps)


def cvar_normal(x, R_block, eps=0.95):
    """Geschlossene CVaR-Formel unter Normalverteilung mit den Stichprobenmomenten des Blocks."""
    k_eps = norm.pdf(norm.ppf(eps)) / (1 - eps)
    r_hat, Sigma_hat = R_block.mean(axis=0), np.cov(R_block.T)
    return -x @ r_hat + k_eps * np.sqrt(x @ Sigma_hat @ x)


# Kontrolle gegen das LP min_gamma F_eps für das gleichgewichtete Portfolio
x_eq = np.full(n, 1 / n)
g, u = cp.Variable(), cp.Variable(S)
lp = cp.Problem(cp.Minimize(g + pi_0 @ u / (1 - eps)), [u >= -R @ x_eq - g, u >= 0])
lp.solve(solver=SOLVER)

Solver: ECOS


np.float64(5.2235864977842486)

### 5.2 RCVaR- und WCVaR-Modell als lineares Programm

Umgesetzt wird $(\mathrm{LP}_{\mathrm{RC}})$ aus Abschnitt 5.2 der Arbeit mit einer eigenen
VaR-Variablen $\gamma_{i}$ und eigenen Hilfsvariablen $u^{i} \in \mathbb{R}^{S}$ je Phase. Die
Renditerestriktion entfällt (siehe Notebook-Abschnitt 6.1).

$$
\begin{aligned}
\min_{x, \gamma, u, \theta} \quad & \theta \\
\text{s.t} \quad & \gamma_{i} + \frac{1}{1-\epsilon} (\pi^{i})^{\top} u^{i} - \psi^{\ast}(\pi^{i}) \leq \theta, \qquad i = 1, 2, 3, \\
& u^{i} \geq -R x - \gamma_{i} e, \quad u^{i} \geq 0, \qquad i = 1, 2, 3, \\
& e^{\top} x = 1, \quad 0 \leq x \leq e.
\end{aligned}
$$

Die Benchmark-Werte werden vorab mit demselben Programm für eine einzelne Phasen berechnet. Setzt man
$\psi^{\ast}(\pi^{i}) = 0$, ergibt sich wieder das WCVaR-Modell bezüglich der endlichen Expertenmenge.

In [6]:
def solve_relative_lp(R, pi_list, psi_star=None, eps=0.95):
    """(LP_RC) ohne Renditerestriktion; für psi_star=None ist psi_i = 0 (WCVaR-Modell)"""
    S, n = R.shape
    psi_star = [0.0] * len(pi_list) if psi_star is None else psi_star
    x, theta = cp.Variable(n), cp.Variable()
    constraints = [cp.sum(x) == 1, x >= 0, x <= 1]
    for pi_i, psi_i in zip(pi_list, psi_star):
        gamma_i, u_i = cp.Variable(), cp.Variable(S)
        constraints += [gamma_i + pi_i @ u_i / (1 - eps) - psi_i <= theta,
                        u_i >= -R @ x - gamma_i, u_i >= 0]
    prob = cp.Problem(cp.Minimize(theta), constraints)
    prob.solve(solver=SOLVER)
    assert prob.status == "optimal", prob.status
    return x.value, prob.value


def solve_benchmark_lp(R, pi_i, eps=0.95):
    """psi*(pi^i): minimaler CVaR unter der einzelnen Verteilung pi^i"""
    return solve_relative_lp(R, [pi_i], eps=eps)[1]


psi_star = [solve_benchmark_lp(R, pi_i, eps=eps) for pi_i in pi_list]
x_WC, val_WC = solve_relative_lp(R, pi_list, eps=eps)
x_RC, val_RC = solve_relative_lp(R, pi_list, psi_star=psi_star, eps=eps)
print("psi*:", dict(zip(pi_experts, np.round(psi_star, 4))))
print(f"WCVaR-Optimalwert: {val_WC:.4f} | RCVaR-Optimalwert (max. Regret): {val_RC:.4f}")

psi*: {'Vorkrise': np.float64(1.6933), 'Crash': np.float64(4.8909), 'Erholung': np.float64(1.4938)}
WCVaR-Optimalwert: 4.8909 | RCVaR-Optimalwert (max. Regret): 0.0745


## 6. Renditerestriktion und Box-Unsicherheit (Abschnitt 7.2.2)

### 6.1 Größte garantierbare Mindestrendite

Die größte Mindestrendite, die in allen drei Phasen zugleich eingehalten werden kann, ist

$$
\mu_{p}^{\max} = \max_{x} \ \min_{i} \ (\pi^{i})^{\top} R x
\quad \text{unter} \quad e^{\top} x = 1, \ 0 \leq x \leq e.
$$

Ist $\mu_{p}^{\max}$ negativ, würde jede zulässige Renditerestriktion einen Verlust in der
Crash-Phase vorgeben und ist als Anlageziel nicht aussagekräftig. Die Zelle rechnet
$\mu_{p}^{\max}$ zusätzlich auf einen Monat um. Eine Restriktion nur für
Zielwerte zwischen der Crash-Rendite von $x^{\mathrm{RC}}$ und $\mu_{p}^{\max}$ wäre sinnvoll. Die Modelle
werden deshalb ohne Renditerestriktion gelöst --> damit gilt
$\mathcal{X} = \mathcal{X}(\pi^{i})$ 

In [7]:
x_v, t_v = cp.Variable(n), cp.Variable()
cp.Problem(cp.Maximize(t_v), [cp.sum(x_v) == 1, x_v >= 0, x_v <= 1]
           + [pi_i @ (R @ x_v) >= t_v for pi_i in pi_list]).solve(solver=SOLVER)
print(f"mu_p^max (in jeder Phase garantierte Rendite): {float(t_v.value):.4f} % pro Tag")
months = len(np.unique(dates // 100))
print(f"Bei {S / months:.1f} Handelstagen je Monat entspricht das rund "
      f"{100 * ((1 + float(t_v.value) / 100) ** round(S / months) - 1):.1f} % pro Monat")
for label, x in [("x_WC", x_WC), ("x_RC", x_RC)]:
    print(f"{label}: mittlere Rendite je Phase", {k: round(float(p @ (R @ x)), 4) for k, p in pi_experts.items()})

mu_p^max (in jeder Phase garantierte Rendite): -0.2457 % pro Tag
Bei 21.1 Handelstagen je Monat entspricht das rund -5.0 % pro Monat
x_WC: mittlere Rendite je Phase {'Vorkrise': 0.0378, 'Crash': -0.2475, 'Erholung': 0.1819}
x_RC: mittlere Rendite je Phase {'Vorkrise': 0.0345, 'Crash': -0.2721, 'Erholung': 0.1874}


### 6.2 Box-Unsicherheitsmenge

Die Box wird wie in Abschnitt 3.4 der Arbeit um $\pi^{0}$ konstruiert,

$$
\underline{\eta}_{k} = -\max_{i} \{\pi^{0}_{k} - \pi^{i}_{k}, 0\} \leq 0,
\qquad
\bar{\eta}_{k} = \max_{i} \{\pi^{i}_{k} - \pi^{0}_{k}, 0\} \geq 0.
$$

Die Zelle berechnet

- die untere Box-Kante $\pi^{0} + \underline{\eta}$,
- den minimalen WCVaR über der endlichen Expertenmenge, über ihrer konvexen Hülle (gemeinsames
  $\gamma$) und über der Box (Risikoteil von $\mathrm{LP}_{\mathrm{B}}$ aus Abschnitt 4.1),
- die größte unter der Box garantierbare Rendite $\max_{x} \min_{\pi \in \mathcal{P}^{B}_{\pi}} \pi^{\top} R x$
  mit der dualisierten Renditerestriktion aus Abschnitt 4.1,
- die Phasen der zehn verlustreichsten Handelstage von $x^{\mathrm{WC}}$. Sie erklären, warum der
  WCVaR über der Box nicht steigt.

In [8]:
P = np.stack(pi_list)
eta_low = -np.max(np.clip(pi_0 - P, 0, None), axis=0)   # <= 0
eta_up = np.max(np.clip(P - pi_0, 0, None), axis=0)     # >= 0
print("Untere Box-Kante pi_0 + eta_low: min/max =", (pi_0 + eta_low).min(), "/", (pi_0 + eta_low).max())


def box_return_terms(x):
    """Dualisierte robuste Rendite min_{pi in Box} pi^T R x (Abschnitt 4.1)"""
    delta, tau, nu = cp.Variable(), cp.Variable(S), cp.Variable(S)
    cons = [np.ones(S) * delta - tau + nu == R @ x, tau >= 0, nu >= 0]
    return pi_0 @ (R @ x) + eta_low @ nu - eta_up @ tau, cons


def solve_wcvar_box():
    """Risikoteil von (LP_B) ohne Renditerestriktion"""
    x, theta, gamma = cp.Variable(n), cp.Variable(), cp.Variable()
    u, kappa, xi, omega = cp.Variable(S), cp.Variable(), cp.Variable(S), cp.Variable(S)
    cons = [cp.sum(x) == 1, x >= 0, x <= 1,
            gamma + pi_0 @ u / (1 - eps) + (eta_up @ xi - eta_low @ omega) / (1 - eps) <= theta,
            np.ones(S) * kappa + xi - omega == u, xi >= 0, omega >= 0,
            u >= -R @ x - gamma, u >= 0]
    prob = cp.Problem(cp.Minimize(theta), cons)
    prob.solve(solver=SOLVER)
    return x.value, prob.value


# WCVaR über der Box
x_box, val_box = solve_wcvar_box()

# WCVaR über der konvexen Hülle: gemeinsame VaR-Variable gamma für alle Phasen
x_h, gamma_h, theta_h = cp.Variable(n), cp.Variable(), cp.Variable()
cons = [cp.sum(x_h) == 1, x_h >= 0, x_h <= 1]
for pi_i in pi_list:
    u_i = cp.Variable(S)
    cons += [gamma_h + pi_i @ u_i / (1 - eps) <= theta_h, u_i >= -R @ x_h - gamma_h, u_i >= 0]
val_hull = cp.Problem(cp.Minimize(theta_h), cons).solve(solver=SOLVER)
print(f"WCVaR endliche Menge: {val_WC:.4f} | konvexe Hülle: {val_hull:.4f} | Box: {val_box:.4f}")

# Größte unter der Box garantierbare Rendite
x_r = cp.Variable(n)
ret, ret_cons = box_return_terms(x_r)
mu_max_box = cp.Problem(cp.Maximize(ret), [cp.sum(x_r) == 1, x_r >= 0, x_r <= 1] + ret_cons).solve(solver=SOLVER)
print(f"Unter der Box garantierbare Rendite: {mu_max_box:.4f} % pro Tag")

# Lage der verlustreichsten Tage von x_WC
phase_of_day = np.empty(S, dtype=object)
for name, m in masks.items():
    phase_of_day[m] = name
worst_days = np.argsort(R @ x_WC)[:10]
print("Phasen der 10 verlustreichsten Tage von x_WC:", pd.Series(phase_of_day[worst_days]).value_counts().to_dict())

Untere Box-Kante pi_0 + eta_low: min/max = 0.0 / 0.0
WCVaR endliche Menge: 4.8909 | konvexe Hülle: 4.8909 | Box: 4.8909
Unter der Box garantierbare Rendite: -1.5522 % pro Tag
Phasen der 10 verlustreichsten Tage von x_WC: {'Crash': 10}


## 7. Normalverteilung und empirisches Tail-Risiko (F1, Abschnitt 7.3.1)

### 7.1 1/n-Portfolio

Für das feste 1/n-Portfolio $x = e/n$, das jeder Branche dasselbe Gewicht gibt, wird der CVaR einmal mit der Normalformel und einmal empirisch
berechnet. Eine negative Abweichung bedeutet, dass die Normalverteilung das Risiko unterschätzt.
Das 99 %-Niveau wird nur für die gesamte Krise berichtet, da innerhalb der Phasen zu wenige
Beobachtungen im Tail liegen (im Crash wäre es etwa ein Handelstag).

In [9]:
pi_all = {"Krise gesamt": pi_0, **pi_experts}
rows = []
for label, pi in pi_all.items():
    m = pi > 0
    for e in ([0.95, 0.99] if label == "Krise gesamt" else [0.95]):
        normal = cvar_normal(x_eq, R[m], e)
        empirical = cvar_discrete(-R @ x_eq, pi, e)
        rows.append({"Zeitraum": label, "eps": e, "CVaR normal": normal, "CVaR empirisch": empirical,
                     "Abweichung normal (%)": 100 * (normal / empirical - 1)})
pd.DataFrame(rows).set_index(["Zeitraum", "eps"]).round(3)

CVaR normal  CVaR empirisch  Abweichung normal (%)
Zeitraum     eps                                                     
Krise gesamt 0.95        4.410           5.224                -15.584
             0.99        5.697           7.835                -27.291
Vorkrise     0.95        2.724           2.828                 -3.661
Crash        0.95        8.259           7.901                  4.542
Erholung     0.95        3.228           3.387                 -4.698

### 7.2 SOCP und LP im Vergleich

Das nominale Modell (eine Verteilung über die gesamte Krise) und das WCVaR-Modell (drei Phasen)
werden jeweils als SOCP unter Normalverteilung (Abschnitt 5.3) und als szenariobasiertes LP
gelöst. Die Spalte *CVaR laut Modell* enthält den Optimalwert, also das Risiko, das das Modell
für sein eigenes Portfolio ausweist. Die Spalte *CVaR empirisch* gibt das tatsächliche Risiko
dieses Portfolios in den Daten an. Für die WCVaR-Modelle ist jeweils der größte CVaR über die drei
Phasen angegeben.

In [10]:
def solve_wcvar_socp(R_blocks, eps=0.95):
    """(SOCP_N) ohne Renditerestriktion mit den Stichprobenmomenten der Blöcke."""
    k_eps = norm.pdf(norm.ppf(eps)) / (1 - eps)
    x, theta = cp.Variable(n), cp.Variable()
    cons = [cp.sum(x) == 1, x >= 0, x <= 1]
    for Rb in R_blocks:
        w, V = np.linalg.eigh(np.cov(Rb.T))
        L = V @ np.diag(np.sqrt(np.clip(w, 0, None)))
        cons.append(-x @ Rb.mean(axis=0) + k_eps * cp.norm(L.T @ x, 2) <= theta)
    prob = cp.Problem(cp.Minimize(theta), cons)
    prob.solve(solver=SOLVER)
    return x.value, prob.value


phase_blocks = [R[m] for m in masks.values()]
x_N_socp, model_N = solve_wcvar_socp([R], eps)          # nominal, eine Verteilung, Normalverteilung
x_N_lp, val_N_lp = solve_relative_lp(R, [pi_0], eps=eps)  # nominal, eine Verteilung, szenariobasiert
x_WC_socp, model_WC = solve_wcvar_socp(phase_blocks, eps)  # WCVaR, drei Phasen, Normalverteilung

price_rows = []
for label, x, model in [("Nominal, SOCP", x_N_socp, model_N), ("Nominal, LP", x_N_lp, val_N_lp)]:
    price_rows.append({"Modell": label, "CVaR laut Modell": model,
                       "CVaR empirisch": cvar_discrete(-R @ x, pi_0, eps)})
for label, x, model in [("WCVaR, SOCP", x_WC_socp, model_WC), ("WCVaR, LP", x_WC, val_WC)]:
    price_rows.append({"Modell": label, "CVaR laut Modell": model,
                       "CVaR empirisch": max(cvar_discrete(-R @ x, p, eps) for p in pi_list)})
pd.DataFrame(price_rows).set_index("Modell").round(4)

,CVaR laut Modell,CVaR empirisch
Modell,,
"Nominal, SOCP",2.7529,3.1792
"Nominal, LP",3.1135,3.1135
"WCVaR, SOCP",5.4103,5.0427
"WCVaR, LP",4.8909,4.8909


### 7.3 Gewichte der optimalen Portfolios (Abschnitte 7.3.1 und 7.3.2)

Gewichte aller optimalen Portfolios in Prozent. Aufgeführt sind die Branchen mit mindestens
0,5 % in einem der Portfolios, der Rest ist unter *Übrige* zusammengefasst. Als Maß für die
Ähnlichkeit zweier Portfolios $x$ und $y$ dient der gemeinsame Anteil
$\sum_{j} \min\{x_{j}, y_{j}\}$. Er ist 1 bei identischen und 0 bei disjunkten Portfolios.

In [11]:
weights = pd.DataFrame({
    "Nominal, SOCP": x_N_socp, "Nominal, LP": x_N_lp, "WCVaR, SOCP": x_WC_socp,
    "WCVaR, LP (x_WC)": x_WC, "RCVaR, LP (x_RC)": x_RC,
}, index=df.columns).clip(lower=0)   # Solver-Rauschen um null abschneiden
shown = (weights >= 0.005).any(axis=1)
weights_tab = 100 * weights[shown]
weights_tab.loc["Übrige"] = 100 * weights[~shown].sum()
display(weights_tab.round(1) + 0.0)


def overlap(x, y):
    """Gemeinsamer Anteil sum_j min(x_j, y_j) zweier Portfolios."""
    return float(np.minimum(np.clip(x, 0, None), np.clip(y, 0, None)).sum())


print(f"Gemeinsamer Anteil SOCP/LP: nominal {overlap(x_N_socp, x_N_lp):.0%} | "
      f"WCVaR {overlap(x_WC_socp, x_WC):.0%}")
print(f"Zwei größte Positionen des nominalen LP zusammen: {np.sort(x_N_lp)[-2:].sum():.0%}")

,"Nominal, SOCP","Nominal, LP","WCVaR, SOCP","WCVaR, LP (x_WC)","RCVaR, LP (x_RC)"
Beer,44.7,77.8,49.7,76.9,53.6
Smoke,20.6,18.3,24.4,6.0,13.5
Hshld,6.5,0.0,2.4,0.0,12.9
Hlth,28.0,0.0,23.6,17.1,19.9
Rtail,0.0,3.9,0.0,0.0,0.0
Übrige,0.2,0.0,0.0,0.0,0.0


Gemeinsamer Anteil SOCP/LP: nominal 63% | WCVaR 73%
Zwei größte Positionen des nominalen LP zusammen: 96%


## 8. Preis des geringeren Regrets (F2, Abschnitt 7.3.2)

Beide Portfolios werden unter jeder Phase ausgewertet. Uns interessiert vor allem, wie viel absolutes
Risiko das RCVaR-Portfolio zusätzlich trägt (Mehrrisiko) und wie viel Regret es dafür einspart
(Regret-Ersparnis).

In [12]:
cvar_WC = np.array([cvar_discrete(-R @ x_WC, p, eps) for p in pi_list])
cvar_RC = np.array([cvar_discrete(-R @ x_RC, p, eps) for p in pi_list])
psi = np.array(psi_star)
phase_df = pd.DataFrame({
    "psi*": psi, "CVaR x_WC": cvar_WC, "Regret x_WC": cvar_WC - psi,
    "CVaR x_RC": cvar_RC, "Regret x_RC": cvar_RC - psi}, index=list(pi_experts))
phase_df.loc["Maximum"] = [np.nan, cvar_WC.max(), (cvar_WC - psi).max(), cvar_RC.max(), (cvar_RC - psi).max()]
display(phase_df.round(4) + 0.0)   # + 0.0 vermeidet die Anzeige -0.0000

print(f"Mehrrisiko x_RC:   {cvar_RC.max() - cvar_WC.max():.4f} ({100 * (cvar_RC.max() / cvar_WC.max() - 1):.1f} %)")
print(f"Regret-Ersparnis:  {(cvar_WC - psi).max() - (cvar_RC - psi).max():.4f} "
      f"({100 * (1 - (cvar_RC - psi).max() / (cvar_WC - psi).max()):.0f} %)")

,psi*,CVaR x_WC,Regret x_WC,CVaR x_RC,Regret x_RC
Vorkrise,1.6933,1.8451,0.1518,1.7677,0.0745
Crash,4.8909,4.8909,0.0000,4.9654,0.0745
Erholung,1.4938,1.6060,0.1122,1.5683,0.0745
Maximum,NaN,4.8909,0.1518,4.9654,0.0745


Mehrrisiko x_RC:   0.0745 (1.5 %)
Regret-Ersparnis:  0.0774 (51 %)


### 8.1 CVaR der einzelnen Branchen (Abschnitt 7.3.2)

Empirischer CVaR jeder Branche für sich (Portfolio aus nur dieser Branche) je Phase und ihr Rang
unter den 30 Branchen (1 = geringster CVaR). Gezeigt werden die Branchen, in die $x^{\mathrm{WC}}$
oder $x^{\mathrm{RC}}$ mindestens 1 % investieren.

In [13]:
branch_cvar = pd.DataFrame({name: [cvar_discrete(-R[:, j], p, eps) for j in range(n)]
                            for name, p in pi_experts.items()}, index=df.columns)
held = df.columns[(x_WC >= 0.01) | (x_RC >= 0.01)]
pd.concat({"CVaR": branch_cvar.loc[held].round(3),
           "Rang": branch_cvar.rank().loc[held].astype(int)}, axis=1)

CVaR                     Rang               
      Vorkrise  Crash Erholung Vorkrise Crash Erholung
Beer     1.966  5.258    1.713        2     1        1
Smoke    2.543  6.533    2.256        5     7        4
Hshld    2.034  6.081    2.457        3     5        7
Hlth     1.955  5.825    1.877        1     3        2

## 9. Robustheit gegenüber den Phasengrenzen (Abschnitte 7.3.1 und 7.3.2)

Neben den Phasen aus Abschnitt 2 (erste Zeile) werden fünf alternative Abgrenzungen betrachtet.
Der Beginn des Crashs wird auf den 01.08.2008 oder den 01.10.2008 verlegt, der Beginn der Erholung
auf den 01.02.2009 oder den 01.05.2009, und in der letzten Variante werden beide Grenzen zugleich
verlegt (01.08.2008 und 01.05.2009). Für jede Variante werden die Abweichung der Normalformel im
Crash (1/n-Portfolio) sowie Mehrrisiko und Regret-Ersparnis des RCVaR-Portfolios neu berechnet.

In [14]:
def phase_variant(crash_start, recovery_start):
    """Kennzahlen für verschobene Phasengrenzen (Format YYYYMMDD)."""
    ms = [dates < crash_start, (dates >= crash_start) & (dates < recovery_start), dates >= recovery_start]
    pis = []
    for m in ms:
        p = np.zeros(S)
        p[m] = 1.0 / m.sum()
        pis.append(p)
    psi_v = np.array([solve_benchmark_lp(R, p, eps=eps) for p in pis])
    xw, _ = solve_relative_lp(R, pis, eps=eps)
    xr, _ = solve_relative_lp(R, pis, psi_star=list(psi_v), eps=eps)
    cw = np.array([cvar_discrete(-R @ xw, p, eps) for p in pis])
    cr = np.array([cvar_discrete(-R @ xr, p, eps) for p in pis])
    crash_normal = cvar_normal(x_eq, R[ms[1]], eps) / cvar_discrete(-R @ x_eq, pis[1], eps) - 1
    return {"Crash: Abweichung normal (%)": 100 * crash_normal,
            "Mehrrisiko x_RC (%)": 100 * (cr.max() / cw.max() - 1),
            "Regret-Ersparnis x_RC (%)": 100 * (1 - (cr - psi_v).max() / (cw - psi_v).max())}


variants = [(20080915, 20090310), (20080801, 20090310), (20081001, 20090310),
            (20080915, 20090201), (20080915, 20090501), (20080801, 20090501)]
robust_df = pd.DataFrame({f"{a} / {b}": phase_variant(a, b) for a, b in variants}).T
robust_df.index.name = "Crash-Beginn / Erholungsbeginn"
display(robust_df.round(1))
print("Spannweiten:", {c: (round(float(robust_df[c].min()), 1), round(float(robust_df[c].max()), 1)) for c in robust_df})

,Crash: Abweichung normal (%),Mehrrisiko x_RC (%),Regret-Ersparnis x_RC (%)
Crash-Beginn / Erholungsbeginn,,,
20080915 / 20090310,4.5,1.5,51.0
20080801 / 20090310,-0.9,1.8,45.7
20081001 / 20090310,5.3,0.2,23.6
20080915 / 20090201,6.4,2.4,64.4
20080915 / 20090501,0.9,1.4,50.2
20080801 / 20090501,-2.5,1.7,52.9


Spannweiten: {'Crash: Abweichung normal (%)': (-2.5, 6.4), 'Mehrrisiko x_RC (%)': (0.2, 2.4), 'Regret-Ersparnis x_RC (%)': (23.6, 64.4)}


### 9.1 Gleich lange Blöcke statt Marktphasen (Abschnitte 7.3.1, 7.3.2 und 8.3)

Zum Vergleich wird die Krise wie in Kapitel 6 ohne Bezug zu Ereignissen in drei gleich lange
Blöcke geteilt

In [15]:
eq_idx = np.array_split(np.arange(S), 3)
eq_crash_start, eq_recovery_start = dates[eq_idx[1][0]], dates[eq_idx[2][0]]
eq_masks = [dates < eq_crash_start,
            (dates >= eq_crash_start) & (dates < eq_recovery_start),
            dates >= eq_recovery_start]

rows = []
for k, m in enumerate(eq_masks, start=1):
    p = np.zeros(S)
    p[m] = 1.0 / m.sum()
    dev = 100 * (cvar_normal(x_eq, R[m], eps) / cvar_discrete(-R @ x_eq, p, eps) - 1)
    rows.append({"Block": k, "Zeitraum": f"{dates[m].min()} - {dates[m].max()}",
                 **normality_summary(df[m]), "Abweichung normal 1/n (%)": round(dev, 1)})
display(pd.DataFrame(rows).set_index("Block"))

print("Gleich lange Blöcke:",
      {k: round(v, 1) for k, v in phase_variant(eq_crash_start, eq_recovery_start).items()})


,Zeitraum,Handelstage,JB verworfen (5 %),Ø Exzess-Kurtosis,Ø Tagesvolatilität,Abweichung normal 1/n (%)
Block,,,,,,
1,20070702 - 20080501,211,20.0%,0.35,1.58,1.2
2,20080502 - 20090304,211,80.0%,2.18,3.32,-8.7
3,20090305 - 20091231,210,83.3%,1.97,2.03,-6.8


Gleich lange Blöcke: {'Crash: Abweichung normal (%)': np.float64(-8.7), 'Mehrrisiko x_RC (%)': np.float64(3.6), 'Regret-Ersparnis x_RC (%)': np.float64(63.4)}
